# Notebook 1: EXPLORING AND CLEANING OLIST E-COMMERCE DATA
-------------------------------------------------------------------------------------------------

### Author: Emmanuel Aregbesola
### Date: April, 2026



In [1]:
# import pandas library

import pandas as pd

In [2]:
# Load the data

files = {
    'customers' : 'olist_customers_dataset.csv',
    'geolocation' : 'olist_geolocation_dataset.csv',
    'order_items' : 'olist_order_items_dataset.csv',
    'payment' : 'olist_order_payments_dataset.csv',
    'reviews' : 'olist_order_reviews_dataset.csv',
    'orders' : 'olist_orders_dataset.csv',
    'products' : 'olist_products_dataset.csv',
    'sellers' : 'olist_sellers_dataset.csv',
    'product_category' : 'product_category_name_translation.csv'
}

data = {}

for name, file in files.items():
    data[name] = pd.read_csv(file)
    
    
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)

In [3]:
# create placeholders for all dataframe

customers_df = data['customers']
geolocation_df = data['geolocation']
order_items_df = data['order_items']
payments_df = data['payment']
reviews_df = data['reviews']
orders_df = data['orders']
products_df = data['products']
sellers_df = data['sellers']
product_category_df = data['product_category']

In [4]:
# basic exploration function 
def basic_exploration(df):
    # 1. Visual Header
    print(f"{'='*30}")
    print("      VITAL STATISTICS")
    print(f"{'='*30}")
    
    # 2. Dimensions and Red Flags
    print(f"Shape: {df.shape}")
    print(f"Duplicate Rows: {df.duplicated().sum()}")
    print("-" * 30)
    
    # 3. Missing Data Profile (Count and Percentage)
    print("MISSING DATA PROFILE:")
    null_count = df.isna().sum()
    null_percent = (null_count / len(df)) * 100
    null_table = pd.concat([null_count, null_percent.round(2)], axis=1)
    null_table.columns = ['Count', 'Percentage (%)']
    print(null_table)
    print("-" * 30)
    
    # 4. Data Types (Clean View)
    print("DATA TYPES:")
    print(df.dtypes)
    print("-" * 30)
    
    # 5. Summary Statistics (Numeric & Categorical)
    print("SUMMARY STATISTICS:")
    # include='all' ensures we see unique counts for strings, not just math for numbers
    display(df.describe(include='all')) 
    print(f"{'='*30}\n")

## Data Cleaning

This section involvces taking each tables individually and cleaning them.

## Payments Dataframe

### order_id: unique identifier of an order.
### payment_sequential: a customer may pay an order with more than one payment method. If he does so, a sequence will be created to
### payment_type: method of payment chosen by the customer.
### payment_installments: number of installments chosen by the customer.
### payment_value: transaction value.

In [5]:
basic_exploration(payments_df)

      VITAL STATISTICS
Shape: (103886, 5)
Duplicate Rows: 0
------------------------------
MISSING DATA PROFILE:
                      Count  Percentage (%)
order_id                  0             0.0
payment_sequential        0             0.0
payment_type              0             0.0
payment_installments      0             0.0
payment_value             0             0.0
------------------------------
DATA TYPES:
order_id                 object
payment_sequential        int64
payment_type             object
payment_installments      int64
payment_value           float64
dtype: object
------------------------------
SUMMARY STATISTICS:


,order_id,payment_sequential,payment_type,payment_installments,payment_value
count,103886,103886.000000,103886,103886.000000,103886.000000
unique,99440,NaN,5,NaN,NaN
top,fa65dad1b0e818e3ccc5cb0e39231352,NaN,credit_card,NaN,NaN
freq,29,NaN,76795,NaN,NaN
mean,NaN,1.092679,NaN,2.853349,154.100380
std,NaN,0.706584,NaN,2.687051,217.494064
min,NaN,1.000000,NaN,0.000000,0.000000
25%,NaN,1.000000,NaN,1.000000,56.790000
50%,NaN,1.000000,NaN,1.000000,100.000000
75%,NaN,1.000000,NaN,4.000000,171.837500


In [12]:
# check basic overview of payments dataframe

payments_df.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [8]:
# Check for frequency for payment type

payments_df['payment_type'].value_counts()

credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: payment_type, dtype: int64

In [11]:
# create a filter dataframe for the not_defined value in the payment_type column

payments_mask = payments_df['payment_type'] == 'not_defined'

payments_df.loc[payments_mask]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0


In [13]:
# check if the order_id for the not_defined values appears in the entire dataframe

problem_id = payments_df.loc[payments_mask, 'order_id']
check_id = payments_df[payments_df['order_id'].isin(problem_id)]
check_id

,order_id,payment_sequential,payment_type,payment_installments,payment_value
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0


In [15]:
# drop the not_defined rows

payments_df = payments_df[~payments_mask].reset_index(drop=True)

In [18]:
# validate the drop

payments_df['payment_type'].value_counts()

credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
Name: payment_type, dtype: int64

In [19]:
payment_zero_mask = payments_df['payment_value'] == 0.0

payments_df.loc[payment_zero_mask]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.0
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.0
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.0
62672,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.0
77883,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.0
100763,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.0


### Summary on payments cleaning

1. After running the basic exploration function on the payments dataframe, it was found that there are no null values, duplicates and the data types are correct. The summary statistics didnt reveal anything out of place except for the fact that there were zero values in the `payment_value` column. 



2. For validity purposes, a value_counts was ran to check what types of payment existed, there it was found that there were `payment_types` tagged `not_defined`. After filtering, it was found that the `not_defined` rows were a sort of ghost records. This was after order_ids that had the `not_defined` values were checked across the dataframe, if there could be another record. They had zero value, and only one payment sequential which led to the rows being dropped.    
Note: These `not_defined` rows were only 3.




3. We also explored the rows that had zero values. It was found that the `payment_value` that had zero were transactions that were paid with vouchers. Since these are legit transactions, they were not dropped.